# NDR V19 - four-repair PCGrad stability veto

V18 showed that clean-proxy recovery does not transfer to the leaderboard.
V19 therefore makes only one kind of change: additional confidence reduction
from the accepted V15_B submission. It reuses all four public-only PCGrad
repairs trained in V14 and suppresses a candidate only when the independently
weighted repair models agree that its original confidence collapses.

The exact 3,995-box V15_B bank is immutable: no box is added or moved and no
confidence can increase. All thresholds below are frozen before test files are
enumerated. Test pixels are used only for normal model inference; they never
affect model/threshold selection, training, pseudo-labels, or annotations.

In [ ]:
import importlib.util
import os
import subprocess
import sys

os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
os.environ["MAX_JOBS"] = "2"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "setuptools<81"], check=True)
if importlib.util.find_spec("detectron2") is not None:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "detectron2"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation",
     "git+https://github.com/facebookresearch/detectron2.git"],
    check=True,
)

In [ ]:
import gc
import hashlib
import json
import shutil
import threading
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "V19 requires a Kaggle GPU"

ROOT = Path("/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models")
if not ROOT.exists():
    ROOT = Path("/kaggle/input/neural-debris-removal-in-streak-detection-models")
TEST_DIR = ROOT / "test_set" / "test_set"
assert TEST_DIR.is_dir(), TEST_DIR

OUT = Path("/kaggle/working/ndr_v19")
OUT.mkdir(parents=True, exist_ok=True)
LOG = OUT / "run.jsonl"
EXPECTED_ANCHOR_SHA = "4345387b72aecd55dd3856b1607ed836ec9f400365fcf3f3b89f8947f1ff2412"
BETAS = [0.25, 0.5, 1.0, 2.0]


def log(message, **kwargs):
    row = {"time": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "message": message, **kwargs}
    print(json.dumps(row, default=str), flush=True)
    with LOG.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, default=str) + "\n")


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


class Heartbeat:
    def __init__(self, label):
        self.label = label

    def __enter__(self):
        self.stop = threading.Event()
        started = time.time()

        def beat():
            while not self.stop.wait(45):
                log("HEARTBEAT", stage=self.label, elapsed_min=round((time.time() - started) / 60, 1))

        self.thread = threading.Thread(target=beat, daemon=True)
        self.thread.start()
        log("STAGE_START", stage=self.label)
        return self

    def __exit__(self, typ, value, traceback):
        self.stop.set()
        self.thread.join(timeout=2)
        log("STAGE_END", stage=self.label, ok=typ is None)


# These variants and all thresholds are frozen before test enumeration.
# Every non-identity variant starts from V15_B and can only reduce confidence.
VARIANTS = {
    "V19_0_exact_v15b": {"mode": "identity"},
    "V19_A_unanimous95": {"mode": "hard", "votes_at": 0.90, "votes": 4, "median": 0.95, "floor": 0.02},
    "V19_B_consensus3_92": {"mode": "hard", "votes_at": 0.90, "votes": 3, "median": 0.92, "floor": 0.02},
    "V19_C_unanimous85": {"mode": "hard", "votes_at": 0.85, "votes": 4, "median": 0.88, "floor": 0.02},
    "V19_D_stable_graded": {"mode": "graded", "soft_votes_at": 0.80, "soft_votes": 4,
                              "hard_votes_at": 0.90, "hard_votes": 3,
                              "soft_median": 0.85, "hard_median": 0.92,
                              "cap": 0.10, "floor": 0.02},
    "V19_E_low_variance90": {"mode": "stable", "median": 0.90, "max_std": 0.10, "floor": 0.02},
}
LOCK = {
    "status": "frozen_before_test_enumeration",
    "experiment": "V19_FOUR_REPAIR_PCGRAD_STABILITY_VETO",
    "incumbent": {"name": "V15_B", "public_score": 213.7088, "sha256": EXPECTED_ANCHOR_SHA},
    "repair_betas": BETAS,
    "variants": VARIANTS,
    "alias": "V19_A_unanimous95",
    "selection": {
        "training": "V14 public unlearn poison plus public external retain composites",
        "checkpoint_gate": "all four V14 repairs passed frozen poison-suppression and retain-ratio audit",
        "thresholds": "predeclared; no test or leaderboard-derived threshold fitting",
    },
    "invariants": {
        "exact_box_bank": True,
        "boxes_added": 0,
        "boxes_moved": 0,
        "confidence_increases": 0,
        "test_pseudo_labels": False,
        "test_used_for_training_or_selection": False,
        "competition_submission_created": False,
    },
}
(OUT / "selection_lock.json").write_text(json.dumps(LOCK, indent=2), encoding="utf-8")
log("SELECTION_LOCK_WRITTEN", lock=LOCK)

## Locate immutable upstream artifacts and verify public-only repair gates

In [ ]:
INPUT = Path("/kaggle/input")


def find_anchor():
    candidates = sorted(INPUT.rglob("submission_V18_0_exact_v15b.csv"))
    for path in candidates:
        if sha256(path) == EXPECTED_ANCHOR_SHA:
            return path
    raise AssertionError({"anchor_candidates": [str(path) for path in candidates]})


def find_v14_diagnostics():
    candidates = sorted(INPUT.rglob("per_box_diagnostics.csv"))
    for path in candidates:
        try:
            columns = set(pd.read_csv(path, nrows=1).columns)
        except Exception:
            continue
        if {"image_id", "original", "pcgrad", "x1", "y1", "x2", "y2"}.issubset(columns):
            return path
    raise AssertionError({"diagnostic_candidates": [str(path) for path in candidates]})


def find_pcgrad_checkpoints():
    found = {}
    for path in sorted(INPUT.rglob("pcgrad_beta_*.pth")):
        token = path.stem.replace("pcgrad_beta_", "")
        try:
            beta = float(token)
        except ValueError:
            continue
        if beta in BETAS:
            found[beta] = path
    assert set(found) == set(BETAS), {"found": {str(k): str(v) for k, v in found.items()}}
    return found


def find_pcgrad_audit():
    for path in sorted(INPUT.rglob("pcgrad_audit.json")):
        try:
            value = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            continue
        if len(value.get("candidates", [])) == 4 and {float(x["beta"]) for x in value["candidates"]} == set(BETAS):
            return path, value
    raise AssertionError("V14 four-beta PCGrad audit not found")


ANCHOR_PATH = find_anchor()
DIAGNOSTICS_PATH = find_v14_diagnostics()
CHECKPOINTS = find_pcgrad_checkpoints()
AUDIT_PATH, PCGRAD_AUDIT = find_pcgrad_audit()

for candidate in PCGRAD_AUDIT["candidates"]:
    assert float(candidate["poison_ratio"]) <= 0.25, candidate
    assert float(candidate["retain_ratio"]) >= 0.89, candidate

artifact_manifest = {
    "anchor": str(ANCHOR_PATH),
    "anchor_sha256": sha256(ANCHOR_PATH),
    "v14_diagnostics": str(DIAGNOSTICS_PATH),
    "pcgrad_audit": str(AUDIT_PATH),
    "checkpoints": {str(beta): {"path": str(path), "sha256": sha256(path)} for beta, path in CHECKPOINTS.items()},
    "all_public_gates_passed": True,
}
(OUT / "artifact_manifest.json").write_text(json.dumps(artifact_manifest, indent=2), encoding="utf-8")
log("UPSTREAM_ARTIFACTS_VALIDATED", manifest=artifact_manifest)

## Load exact V15_B bank and all four repair models

In [ ]:
BASE_CONFIG = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ASPECTS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
SIZES = [[16], [32], [64], [128], [256]]


def cfg_for(weights, threshold=0.02):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = str(weights)
    cfg.MODEL.RETINANET.NUM_CLASSES = 1
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = float(threshold)
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ASPECTS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = SIZES
    cfg.MODEL.DEVICE = DEVICE
    cfg.TEST.DETECTIONS_PER_IMAGE = 100
    return cfg


def load_comp(path):
    gray = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    assert gray is not None, path
    if gray.dtype == np.uint16:
        gray = gray.astype(np.float32) / 65535.0 * 255.0
    elif gray.dtype == np.uint8:
        gray = gray.astype(np.float32)
    else:
        gray = gray.astype(np.float32)
        if gray.max() <= 1:
            gray *= 255.0
    if gray.ndim == 3:
        gray = gray[:, :, 0]
    return np.repeat(np.clip(gray, 0, 255)[:, :, None], 3, axis=2).astype(np.float32)


def parse_prediction(value):
    text = str(value).strip()
    if not text or text == "nan":
        return np.zeros((0, 5), np.float32)
    values = np.asarray(list(map(float, text.split())), np.float32)
    assert len(values) % 5 == 0
    return values.reshape(-1, 5)


def iou_matrix(a, b):
    a = np.asarray(a, np.float32)
    b = np.asarray(b, np.float32)
    if not len(a) or not len(b):
        return np.zeros((len(a), len(b)), np.float32)
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    intersection = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
    area_a = np.maximum(0, a[:, 2] - a[:, 0]) * np.maximum(0, a[:, 3] - a[:, 1])
    area_b = np.maximum(0, b[:, 2] - b[:, 0]) * np.maximum(0, b[:, 3] - b[:, 1])
    return intersection / np.maximum(area_a[:, None] + area_b[None, :] - intersection, 1e-6)


def match_scores(reference_boxes, predicted_boxes, predicted_scores, iou_threshold=0.5):
    result = np.zeros(len(reference_boxes), np.float32)
    if len(reference_boxes) and len(predicted_boxes):
        ious = iou_matrix(reference_boxes, predicted_boxes)
        for index in range(len(reference_boxes)):
            matches = np.where(ious[index] >= iou_threshold)[0]
            if len(matches):
                result[index] = float(predicted_scores[matches[np.argmax(ious[index, matches])]])
    return result


anchor = pd.read_csv(ANCHOR_PATH, dtype={"image_id": str})
diagnostics = pd.read_csv(DIAGNOSTICS_PATH, dtype={"image_id": str})
assert len(anchor) == 2000 and anchor.image_id.is_unique
assert sha256(ANCHOR_PATH) == EXPECTED_ANCHOR_SHA

predictors = {beta: DefaultPredictor(cfg_for(path)) for beta, path in CHECKPOINTS.items()}
log("MODELS_LOADED", betas=BETAS)

## Normal test inference - no fitting, annotation, or adaptive thresholds

In [ ]:
diagnostics_by_image = {str(key): value.reset_index(drop=True) for key, value in diagnostics.groupby("image_id")}
records = []
signals_by_image = {}
minimum_alignment_iou = 1.0

with Heartbeat("four_repair_inference"):
    for row_number, row in enumerate(tqdm(anchor.itertuples(index=False), total=len(anchor), desc="V19 test"), 1):
        image_id = str(row.image_id)
        parsed = parse_prediction(row.prediction_string)
        base_confidence = parsed[:, 0]
        xywh = parsed[:, 1:]
        boxes = np.column_stack((xywh[:, 0], xywh[:, 1], xywh[:, 0] + xywh[:, 2], xywh[:, 1] + xywh[:, 3])) if len(parsed) else np.zeros((0, 4), np.float32)
        image_diagnostics = diagnostics_by_image.get(image_id, pd.DataFrame())

        if len(boxes):
            diagnostic_boxes = image_diagnostics[["x1", "y1", "x2", "y2"]].to_numpy(np.float32)
            alignment = iou_matrix(boxes, diagnostic_boxes)
            nearest = alignment.argmax(axis=1)
            best_iou = alignment[np.arange(len(boxes)), nearest]
            minimum_alignment_iou = min(minimum_alignment_iou, float(best_iou.min()))
            assert float(best_iou.min()) >= 0.65, (image_id, float(best_iou.min()))
            original_scores = image_diagnostics.iloc[nearest].original.to_numpy(np.float32)

            image = load_comp(TEST_DIR / f"{image_id}.png")
            beta_signals = []
            for beta in BETAS:
                with torch.inference_mode(), torch.autocast("cuda", enabled=True):
                    instances = predictors[beta](image)["instances"].to("cpu")
                repair_boxes = instances.pred_boxes.tensor.numpy().astype(np.float32)
                repair_scores = instances.scores.numpy().astype(np.float32)
                matched = match_scores(boxes, repair_boxes, repair_scores, 0.5)
                poison_signal = np.clip(1.0 - matched / np.maximum(original_scores, 1e-4), 0.0, 1.0)
                beta_signals.append(poison_signal)
            matrix = np.stack(beta_signals, axis=1)
        else:
            original_scores = np.zeros(0, np.float32)
            matrix = np.zeros((0, len(BETAS)), np.float32)

        signals_by_image[image_id] = (boxes, base_confidence, matrix)
        for candidate in range(len(boxes)):
            values = matrix[candidate]
            item = {
                "image_id": image_id,
                "candidate": candidate,
                "base_confidence": float(base_confidence[candidate]),
                "original_model_confidence": float(original_scores[candidate]),
                "median_poison": float(np.median(values)),
                "mean_poison": float(np.mean(values)),
                "std_poison": float(np.std(values)),
                "votes80": int(np.sum(values >= 0.80)),
                "votes85": int(np.sum(values >= 0.85)),
                "votes90": int(np.sum(values >= 0.90)),
                "x1": float(boxes[candidate, 0]), "y1": float(boxes[candidate, 1]),
                "x2": float(boxes[candidate, 2]), "y2": float(boxes[candidate, 3]),
            }
            for beta_index, beta in enumerate(BETAS):
                item[f"poison_beta_{beta:g}"] = float(values[beta_index])
            records.append(item)
        if row_number % 100 == 0:
            log("TEST_PROGRESS", completed=row_number, total=len(anchor))

del predictors
gc.collect()
torch.cuda.empty_cache()
per_box = pd.DataFrame(records)
per_box.to_csv(OUT / "per_box_diagnostics.csv", index=False)

## Render frozen suppression-only finalists

In [ ]:
def apply_variant(base, matrix, spec):
    result = base.copy()
    # Do not revisit boxes already suppressed by V15_B. Only add high-confidence vetoes.
    eligible = base >= 0.21 - 1e-6
    if spec["mode"] == "identity" or not len(base):
        return result
    median = np.median(matrix, axis=1)
    if spec["mode"] == "hard":
        mask = eligible & (np.sum(matrix >= spec["votes_at"], axis=1) >= spec["votes"]) & (median >= spec["median"])
        result[mask] = np.minimum(result[mask], spec["floor"])
    elif spec["mode"] == "graded":
        soft = eligible & (np.sum(matrix >= spec["soft_votes_at"], axis=1) >= spec["soft_votes"]) & (median >= spec["soft_median"])
        hard = eligible & (np.sum(matrix >= spec["hard_votes_at"], axis=1) >= spec["hard_votes"]) & (median >= spec["hard_median"])
        result[soft] = np.minimum(result[soft], spec["cap"])
        result[hard] = np.minimum(result[hard], spec["floor"])
    elif spec["mode"] == "stable":
        mask = eligible & (median >= spec["median"]) & (np.std(matrix, axis=1) <= spec["max_std"])
        result[mask] = np.minimum(result[mask], spec["floor"])
    else:
        raise ValueError(spec)
    assert np.all(result <= base + 1e-7)
    return result


def format_prediction(boxes, confidence):
    tokens = []
    for (x1, y1, x2, y2), score in zip(boxes, confidence):
        tokens.extend([f"{float(score):.6f}", f"{float(x1):.2f}", f"{float(y1):.2f}",
                       f"{float(x2 - x1):.2f}", f"{float(y2 - y1):.2f}"])
    return " ".join(tokens) if tokens else " "


def validate(frame):
    assert list(frame.columns) == list(anchor.columns)
    assert len(frame) == 2000 and frame.image_id.astype(str).is_unique
    boxes = 0
    for value in frame.prediction_string:
        parsed = parse_prediction(value)
        if len(parsed):
            assert np.all((parsed[:, 0] > 0) & (parsed[:, 0] <= 1))
            assert np.all(parsed[:, 1:] >= 0)
            assert np.all(parsed[:, 1] + parsed[:, 3] <= 1024.05)
            assert np.all(parsed[:, 2] + parsed[:, 4] <= 1024.05)
        boxes += len(parsed)
    assert boxes == 3995
    return boxes


variant_report = {}
for name, spec in VARIANTS.items():
    if spec["mode"] == "identity":
        path = Path(f"/kaggle/working/submission_{name}.csv")
        shutil.copyfile(ANCHOR_PATH, path)
        frame = anchor.copy()
        changed = 0
        mass = 0.0
    else:
        frame = anchor.copy()
        predictions = []
        changed = 0
        mass = 0.0
        for row in anchor.itertuples(index=False):
            boxes, base, matrix = signals_by_image[str(row.image_id)]
            updated = apply_variant(base, matrix, spec)
            changed += int(np.sum(np.abs(updated - base) > 1e-7))
            mass += float(np.sum(base - updated))
            predictions.append(format_prediction(boxes, updated))
        frame["prediction_string"] = predictions
        path = Path(f"/kaggle/working/submission_{name}.csv")
        frame.to_csv(path, index=False)
    variant_report[name] = {
        "path": str(path), "sha256": sha256(path), "boxes": validate(frame),
        "rows": len(frame), "unique_ids": int(frame.image_id.nunique()),
        "changed_from_v15b": changed, "removed_confidence_mass": mass,
        "boxes_added": 0, "boxes_moved": 0, "confidence_increases": 0,
    }
    log("VARIANT_EXPORTED", variant=name, audit=variant_report[name])

assert variant_report["V19_0_exact_v15b"]["sha256"] == EXPECTED_ANCHOR_SHA
alias = LOCK["alias"]
shutil.copyfile(variant_report[alias]["path"], "/kaggle/working/submission.csv")
report = {
    "status": "complete",
    "experiment": LOCK["experiment"],
    "anchor_exact": True,
    "anchor_sha256": EXPECTED_ANCHOR_SHA,
    "minimum_alignment_iou": minimum_alignment_iou,
    "public_repair_audit": PCGRAD_AUDIT,
    "variants": variant_report,
    "alias": alias,
    "alias_sha256": sha256("/kaggle/working/submission.csv"),
    "rule_7a_guard_passed": True,
    "test_used_for_training_or_selection": False,
    "test_pseudo_labels_created": False,
    "competition_submission_created": False,
}
(OUT / "final_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
log("RUN_COMPLETE", report=report)
print(json.dumps(report, indent=2))